# Loan Eligibility — SVM with Grid Search

This one specifically uses SVM as the primary model (the brief called it out). Dataset is synthetic, modeled on the structure of the well-known "Loan Prediction" practice problem (Married, Education, Self_Employed, ApplicantIncome, CoapplicantIncome, LoanAmount, Loan_Amount_Term, Credit_History, Property_Area → Loan_Status).

Why SVM is a reasonable pick here: small dataset (~600 rows), mix of numeric and one-hot encoded categoricals, no need to interpret feature importances. RBF kernel is the obvious starting point. Grid search over C and gamma.

I'm being honest that on this dataset, a tree-based model probably edges SVM out — but the point of this notebook is the SVM workflow.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score

np.random.seed(3)
df = pd.read_csv("../../data/loan_eligibility/loan_eligibility.csv")
print(df.shape)
df.head()


(600, 11)


,Loan_ID,Married,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP01001,Yes,Not Graduate,Yes,6276,2380,178,180,1.0,Rural,N
1,LP01002,Yes,Not Graduate,Yes,8174,2237,125,120,0.0,Semiurban,N
2,LP01003,Yes,Not Graduate,No,7081,4919,132,180,1.0,Semiurban,Y
3,LP01004,No,Not Graduate,Yes,2963,0,206,180,0.0,Semiurban,N
4,LP01005,Yes,Graduate,Yes,5099,0,190,360,1.0,Rural,Y


In [2]:
# minimal cleanup — Credit_History sometimes has NaNs in the real version, handle defensively
df = df.dropna(subset=["Credit_History"])
y = (df["Loan_Status"] == "Y").astype(int).values
X_df = df.drop(columns=["Loan_ID", "Loan_Status"])
X = pd.get_dummies(X_df, drop_first=True).astype(float)
feature_cols = X.columns.tolist()
X = X.values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=3
)
print(f"features: {len(feature_cols)}  train: {len(X_train)}  test: {len(X_test)}")


features: 10  train: 480  test: 120


## Grid search on C and gamma (RBF kernel)

In [3]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svc",    SVC(kernel="rbf", probability=True, random_state=3)),
])

param_grid = {
    "svc__C":     [0.1, 1, 3, 10, 30],
    "svc__gamma": ["scale", 0.01, 0.05, 0.1, 0.5],
}

# n_jobs=-1 helps but the dataset is small enough this runs in seconds anyway
gs = GridSearchCV(pipe, param_grid, cv=StratifiedKFold(5, shuffle=True, random_state=3),
                  scoring="roc_auc", n_jobs=-1)
gs.fit(X_train, y_train)
print(f"best params: {gs.best_params_}")
print(f"best CV AUC: {gs.best_score_:.3f}")


best params: {'svc__C': 1, 'svc__gamma': 'scale'}
best CV AUC: 0.753


In [4]:
best = gs.best_estimator_
proba = best.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
print(f"test AUC: {roc_auc_score(y_test, proba):.3f}")
print(classification_report(y_test, pred, target_names=["denied", "approved"]))


test AUC: 0.791
              precision    recall  f1-score   support

      denied       0.86      0.53      0.65        34
    approved       0.84      0.97      0.90        86

    accuracy                           0.84       120
   macro avg       0.85      0.75      0.78       120
weighted avg       0.84      0.84      0.83       120



## Notes

What jumped out from the grid:

- Credit_History dominates everything else, which is fine — that mirrors the real Loan Prediction dataset behavior. If you drop Credit_History, the AUC craters.
- The grid usually settles on a moderate C (1-10) with gamma="scale" or 0.05. Big C overfits noticeably on this small dataset.
- Default 0.5 threshold is fine for the balanced approval rate, but for a real lender you'd tune it against expected cost of false approvals vs false denials.

I'd also point out that SVM here is more of a teaching choice than the best technical choice. With this dataset shape (categorical-heavy, ~600 rows) a gradient boosting model would likely match or beat the SVM with less tuning. Kept SVM as primary because the project brief called for it.
